In [1]:
import pandas as pd

In [2]:
pd.set_option("display.max_columns", 30)

In [3]:
df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/silver/flight_information/flights_info_silver.parquet")

## Feature Extraction 

### Departure Lat & Lon & Altitude

In [4]:
airport_df = pd.read_csv("/Users/YGT/ist-airport-decision-support-system/data/meta/airportdata.txt")

In [5]:
airport_df.shape

(84654, 19)

In [6]:
airport_df = airport_df[["name", "latitude_deg","longitude_deg", "elevation_ft", "icao_code"]]

In [7]:
set_data = set(df["dep_code_icao"].unique())
set_airports = set(airport_df["icao_code"].unique())

# Kesişim (Eşleşenler)
matched = set_data.intersection(set_airports)

# Fark (Eşleşmeyenler - senin verinde olup kütüphanede olmayanlar)
missing = set_data.difference(set_airports)

print("="*40)
print(f"Flight İnfo Dataset Unique ICAO: {len(set_data):,}")
print(f"Airport Dataset Unique ICAO: {len(set_airports):,}")
print("-"*40)
print(f"Matched ICAO Count: {len(matched):,}")
print(f"Missing ICAO Count: {len(missing):,}")
print(f"Matched Ratio: %{(len(matched)/len(set_data))*100:.2f}")
print("="*40)

if len(missing) > 0:
    print("\n not matched icao code :")
    print(list(missing))

Flight İnfo Dataset Unique ICAO: 341
Airport Dataset Unique ICAO: 9,730
----------------------------------------
Matched ICAO Count: 328
Missing ICAO Count: 13
Matched Ratio: %96.19

 not matched icao code :
['UTTT', 'UTFF', 'UTFA', 'HEBA', 'LERJ', 'OSNH', 'HSSJ', 'LTST', 'UTSS', 'UTNU', 'UAFM', 'UTSB', 'OKBK']


In [8]:
airport_df.columns

Index(['name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'icao_code'], dtype='str')

In [9]:
airport_lookup = airport_df.copy()

df = pd.merge(
    df,
    airport_lookup,
    left_on='dep_code_icao',
    right_on='icao_code',
    how='left'
)

In [10]:
df.head(2)

,date,day_of_week,hour_ist,hex_icao,aircraft_type,aircraft_registration,airline_name_english,callsign_code_iata,callsign_code_icao,airline_iata,airline_icao,dep_code_iata,dep_code_icao,dep_name_english,dest_code_iata,dest_code_icao,dest_name_english,dest_lat,dest_lon,dest_altitude,arr_sched_time_utc,arr_revised_time_utc,status,name,latitude_deg,longitude_deg,elevation_ft,icao_code
0,2025-07-31,Thursday,21,728678,Airbus A320,YI-ASX,I A W,IA 223,IAW223,IA,IAW,BGW,ORBI,Baghdad,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:00:00+00:00,2025-07-31 18:00:00+00:00,Arrived,Baghdad International Airport / New Al Muthana...,33.262501,44.234600,114.0,ORBI
1,2025-07-31,Thursday,21,7114EF,Airbus A321,UNKNOWN,Saudi Arabian,SV 261,SVA261,SV,SVA,JED,OEJN,Jeddah,IST,LTFM,Istanbul Airport,41.2751,28.7519,325,2025-07-31 18:10:00+00:00,2025-07-31 18:10:00+00:00,Arrived,King Abdulaziz International Airport,21.680241,39.157436,48.0,OEJN


In [11]:
df['dep_lat'] = df['latitude_deg'].astype(float)
df['dep_lon'] = df['longitude_deg'].astype(float)
df['dep_altitude'] = df['elevation_ft'].astype(float) #ft

In [12]:
cols_to_drop = ['name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'icao_code']

df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [13]:
df.loc[
    (df["dep_lat"] == 0.0) &
    (df["dep_lon"] == 0.0)

]

,date,day_of_week,hour_ist,hex_icao,aircraft_type,aircraft_registration,airline_name_english,callsign_code_iata,callsign_code_icao,airline_iata,airline_icao,dep_code_iata,dep_code_icao,dep_name_english,dest_code_iata,dest_code_icao,dest_name_english,dest_lat,dest_lon,dest_altitude,arr_sched_time_utc,arr_revised_time_utc,status,dep_lat,dep_lon,dep_altitude


In [14]:
['UTFF', 'OKBK', 'UTSB', 'HEBA', 'UTSS', 'HSSJ', 'LTST', 'UTTT', 'UTNU', 'UTFA', 'UAFM', 'LERJ', 'OSNH']

['UTFF',
 'OKBK',
 'UTSB',
 'HEBA',
 'UTSS',
 'HSSJ',
 'LTST',
 'UTTT',
 'UTNU',
 'UTFA',
 'UAFM',
 'LERJ',
 'OSNH']

In [15]:
AIRPORT_PATCH = [
    {"icao": "UTFF", "lat": 37.2867, "lon": 67.3100, "alt": 1017.0},
    {"icao": "UTNU", "lat": 42.4872, "lon": 59.6231, "alt": 250.0},
    {"icao": "OSNH", "lat": 37.0219, "lon": 41.1933, "alt": 1483.0},
    {"icao": "HSSJ", "lat": 4.8720,  "lon": 31.6011, "alt": 1457.0},
    {"icao": "LTST", "lat": 37.3647, "lon": 42.0583, "alt": 2038.0},
    {"icao": "UTTT", "lat": 41.2579, "lon": 69.2812, "alt": 1417.0},
    {"icao": "OKBK", "lat": 29.2266, "lon": 47.9689, "alt": 206.0},
    {"icao": "UTSB", "lat": 39.7750, "lon": 64.4833, "alt": 751.0},
    {"icao": "HEBA", "lat": 30.9177, "lon": 29.6964, "alt": 177.0},
    {"icao": "UTSS", "lat": 39.7005, "lon": 66.9838, "alt": 2224.0},
    {"icao": "UTFA", "lat": 40.7269, "lon": 72.2925, "alt": 1558.0},
    {"icao": "UAFM", "lat": 43.0613, "lon": 74.4776, "alt": 2058.0},
    {"icao": "LERJ", "lat": 42.4606, "lon": -2.3206, "alt": 1156.0},
]

In [16]:
patch_df = pd.DataFrame(AIRPORT_PATCH).copy()

patch_df = patch_df.rename(columns={
    "icao": "dep_code_icao",
    "lat": "dep_lat_patch",
    "lon": "dep_lon_patch",
    "alt": "dep_altitude_patch",
})


patch_df = patch_df[["dep_code_icao", "dep_lat_patch", "dep_lon_patch", "dep_altitude_patch"]]

df = df.merge(patch_df, on="dep_code_icao", how="left")

# fill only missing
df["dep_lat"] = df["dep_lat"].fillna(df["dep_lat_patch"])
df["dep_lon"] = df["dep_lon"].fillna(df["dep_lon_patch"])
df["dep_altitude"] = df["dep_altitude"].fillna(df["dep_altitude_patch"])

df = df.drop(columns=["dep_lat_patch", "dep_lon_patch", "dep_altitude_patch"])

In [17]:
df.isna().sum()

date                      0
day_of_week               0
hour_ist                  0
hex_icao                  0
aircraft_type             0
aircraft_registration     0
airline_name_english      0
callsign_code_iata        0
callsign_code_icao        0
airline_iata              0
airline_icao              0
dep_code_iata             0
dep_code_icao             0
dep_name_english          0
dest_code_iata            0
dest_code_icao            0
dest_name_english         0
dest_lat                  0
dest_lon                  0
dest_altitude             0
arr_sched_time_utc        0
arr_revised_time_utc     38
status                    0
dep_lat                   0
dep_lon                   0
dep_altitude              0
dtype: int64

### Great Circle Distance

In [18]:
import numpy as np

In [19]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df["distance"] = haversine_km(
    df["dep_lat"].astype(float),
    df["dep_lon"].astype(float),
    df["dest_lat"].astype(float),
    df["dest_lon"].astype(float)
).astype("float32")

### Haul

In [20]:
def assign_haul_wragg(dist):
    if dist < 1600:
        return 'short-haul'
    elif 1600 <= dist <= 4000:
        return 'medium-haul'
    else:
        return 'long-haul'

df['haul'] = df['distance'].apply(assign_haul_wragg)

### Wake Turbulence Category

In [21]:
df.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'arr_revised_time_utc', 'status', 'dep_lat',
       'dep_lon', 'dep_altitude', 'distance', 'haul'],
      dtype='str')

In [22]:
unique_aircraft = df['aircraft_type'].unique()
unique_aircraft

<ArrowStringArray>
[                'Airbus A320',                 'Airbus A321',
                  'Boeing 737',              'Boeing 737-800',
              'Boeing 737-900',                'Boeing 787-9',
             'Airbus A350-900',             'Airbus A330-300',
             'Airbus A330-200',             'Airbus A320 NEO',
            'Boeing 777-300ER',         'Sukhoi Superjet 100',
                 'Airbus A330',                 'Airbus A319',
              'Boeing 767-300',              'Boeing 777-300',
                 'Airbus A300',                 'Embraer 195',
              'Boeing 737-300',                  'Boeing 777',
                'Boeing 787-8',             'Airbus A220-300',
             'Airbus A321 NEO',                 'Airbus A340',
             'Airbus A340-300',                 'Embraer 190',
             'Airbus A300-600',     'Airbus A321 (Sharklets)',
                      'ATR 72',             'Airbus A310-300',
     'Airbus A320 (Sharklets)',     

In [23]:
WTC = {
  "Airbus A300":"H","Airbus A300-600":"H","Airbus A300-600H":"H",
  "Airbus A310":"H","Airbus A310-300":"H",
  "Airbus A330":"H","Airbus A330-200":"H","Airbus A330-300":"H","Airbus A330-900":"H",
  "Airbus A340":"H","Airbus A340-300":"H","Airbus A340-600":"H",
  "Airbus A350":"H","Airbus A350-900":"H",
  "Boeing 747":"H",
  "Boeing 767":"H","Boeing 767-200":"H","Boeing 767-300":"H",
  "Boeing 777":"H","Boeing 777-200":"H","Boeing 777-200LR":"H","Boeing 777-300":"H","Boeing 777-300ER":"H",
  "Boeing 787":"H","Boeing 787-8":"H","Boeing 787-9":"H",
  "Airbus A380-800":"H",


  "Airbus":"M",
  "Airbus A319":"M","Airbus A320":"M","Airbus A320 NEO":"M","Airbus A320 (Sharklets)":"M",
  "Airbus A321":"M","Airbus A321 NEO":"M","Airbus A321-200":"M","Airbus A321-200 NEO":"M",
  "Airbus A321-200 (sharklets)":"M","Airbus A321-200 (Sharklets)":"M",
  "Airbus A321 (sharklets)":"M","Airbus A321 (Sharklets)":"M",
  "Airbus A321-211":"M","Airbus A321 (Sharklets)":"M",
  "Airbus A220-300":"M",
  "Boeing 737":"M","Boeing 737-300":"M","Boeing 737-400":"M","Boeing 737-500":"M","Boeing 737-700":"M",
  "Boeing 737-800":"M","Boeing 737-800 (winglets)":"M","Boeing 737-900":"M",
  "Boeing 737 MAX 8":"M","Boeing 737-8LJ":"M",
  "Boeing 727-200 Freighter":"M",
  "Embraer 175":"M","Embraer 190":"M","Embraer 195":"M","Embraer RJ145":"M",
  "Bombardier CRJ":"M","Bombardier CRJ1000":"M",
  "ATR 72":"M","BAe 146-300":"M",
  "Sukhoi Superjet 100":"M","Sukhoi Superjet 100-95":"M",
  "35D":"M",

  "W38":"L",
}

df["wake_turbulence_cat"] = df["aircraft_type"].map(WTC)

In [24]:
df["wake_turbulence_cat"].isna().sum()

np.int64(0)

### Actual Entry Time (ADS-B)

In [25]:
traj_df = pd.read_parquet("/Users/YGT/ist-airport-decision-support-system/data/gold/trajectories/trajectory_gold.parquet")

In [26]:
traj_df.head()

,flight_id,episode_id,point_timestamp,icao,date,e_m,n_m,u_m,ux,uy,uz,r,sin_theta,cos_theta,delta_t,distance_km,segment_id,gap_flag
0,000001_0,0,2025-03-10 09:07:51+00:00,000001,2025-03-10,-0.225343,-0.550454,0.064425,0.959042,-0.283260,0.001077,0.594793,-0.925454,-0.378859,0.000000,71.336296,1,0
1,000001_0,0,2025-03-10 09:07:55+00:00,000001,2025-03-10,-0.221211,-0.551674,0.064430,0.929566,-0.368655,0.000012,0.594372,-0.928162,-0.372176,1.609438,71.287247,1,0
2,000001_0,0,2025-03-10 09:07:57+00:00,000001,2025-03-10,-0.219448,-0.552373,0.064430,0.902580,-0.430521,-0.000794,0.594368,-0.929345,-0.369213,1.098612,71.287399,1,0
3,000001_0,0,2025-03-10 09:08:00+00:00,000001,2025-03-10,-0.216611,-0.553726,0.064427,0.898086,-0.439818,-0.001030,0.594587,-0.931279,-0.364306,1.386294,71.314766,1,0
4,000001_0,0,2025-03-10 09:08:09+00:00,000001,2025-03-10,-0.207627,-0.558126,0.064417,0.937753,-0.347303,-0.000051,0.595495,-0.937248,-0.348663,2.302585,71.426468,1,0


In [27]:
traj_df = traj_df[["flight_id", "point_timestamp", "icao", "date", "distance_km"]]
traj_df.head()

,flight_id,point_timestamp,icao,date,distance_km
0,000001_0,2025-03-10 09:07:51+00:00,000001,2025-03-10,71.336296
1,000001_0,2025-03-10 09:07:55+00:00,000001,2025-03-10,71.287247
2,000001_0,2025-03-10 09:07:57+00:00,000001,2025-03-10,71.287399
3,000001_0,2025-03-10 09:08:00+00:00,000001,2025-03-10,71.314766
4,000001_0,2025-03-10 09:08:09+00:00,000001,2025-03-10,71.426468


In [28]:
traj_df = traj_df.sort_values(
    ["icao","point_timestamp"],
    kind="mergesort"
).reset_index(drop=True)

In [29]:
traj_df.head()

,flight_id,point_timestamp,icao,date,distance_km
0,000001_0,2025-03-10 09:07:51+00:00,000001,2025-03-10,71.336296
1,000001_0,2025-03-10 09:07:55+00:00,000001,2025-03-10,71.287247
2,000001_0,2025-03-10 09:07:57+00:00,000001,2025-03-10,71.287399
3,000001_0,2025-03-10 09:08:00+00:00,000001,2025-03-10,71.314766
4,000001_0,2025-03-10 09:08:09+00:00,000001,2025-03-10,71.426468


In [30]:
first_seen = (
    traj_df
    .sort_values(["flight_id", "point_timestamp"], kind="mergesort")
    .groupby("flight_id", sort=False)
    .agg(
        actual_entry_time = ("point_timestamp", "first"),
        date              = ("date", "first"),
        icao              = ("icao", "first"),
        distance_km       = ("distance_km", "first"),
    )
    .reset_index()
)

In [31]:
first_seen["distance_km"].describe()

count    566791.000000
mean         66.222015
std          36.246704
min           0.292297
25%          33.334629
50%          64.835800
75%          99.009083
max         119.999992
Name: distance_km, dtype: float64

In [32]:
len(first_seen)

566791

In [34]:
df.columns

Index(['date', 'day_of_week', 'hour_ist', 'hex_icao', 'aircraft_type',
       'aircraft_registration', 'airline_name_english', 'callsign_code_iata',
       'callsign_code_icao', 'airline_iata', 'airline_icao', 'dep_code_iata',
       'dep_code_icao', 'dep_name_english', 'dest_code_iata', 'dest_code_icao',
       'dest_name_english', 'dest_lat', 'dest_lon', 'dest_altitude',
       'arr_sched_time_utc', 'arr_revised_time_utc', 'status', 'dep_lat',
       'dep_lon', 'dep_altitude', 'distance', 'haul', 'wake_turbulence_cat'],
      dtype='str')

In [33]:
first_seen.head()

,flight_id,actual_entry_time,date,icao,distance_km
0,000001_0,2025-03-10 09:07:51+00:00,2025-03-10,000001,71.336296
1,000001_1,2025-03-10 13:40:11+00:00,2025-03-10,000001,100.560425
2,000001_4,2025-03-14 11:05:42+00:00,2025-03-14,000001,74.398598
3,000016_0,2025-10-08 04:31:55+00:00,2025-10-08,000016,22.090853
4,00006a_0,2025-03-10 15:00:37+00:00,2025-03-10,00006a,31.739309


In [35]:
first_seen["icao"] = first_seen["icao"].str.strip().str.upper()
df["hex_icao"] = df["hex_icao"].str.strip().str.upper()

In [37]:
df        = df.sort_values("arr_sched_time_utc")
first_seen= first_seen.sort_values("actual_entry_time")

In [ ]:
df_merged = pd.merge_asof(
    df,
    first_seen[["icao", "actual_entry_time"]],
    left_on       = "arr_sched_time_utc",
    right_on      = "actual_entry_time",
    left_by       = "hex_icao",
    right_by      = "icao",
    direction     = "nearest",
    tolerance     = pd.Timedelta("1D")
).drop(columns="icao")

,flight_id,actual_entry_time,date,icao,distance_km
488923,4d2490_0,2025-03-01 00:06:15+00:00,2025-03-01,4D2490,118.922089
479030,4d2003_0,2025-03-01 00:14:14+00:00,2025-03-01,4D2003,117.131554
565799,abd336_0,2025-03-01 00:15:52+00:00,2025-03-01,ABD336,116.810753
25418,392af0_0,2025-03-01 00:25:14+00:00,2025-03-01,392AF0,118.533340
63980,48af01_0,2025-03-01 00:27:18+00:00,2025-03-01,48AF01,118.714371
...,...,...,...,...,...
78300,4a29f9_337,2026-01-31 23:22:09+00:00,2026-01-31,4A29F9,119.975266
543534,78149e_59,2026-01-31 23:42:52+00:00,2026-01-31,78149E,114.255562
74225,4a08e6_215,2026-01-31 23:45:38+00:00,2026-01-31,4A08E6,119.533020
8820,040168_11,2026-01-31 23:47:50+00:00,2026-01-31,040168,118.535667
